# Chronological research and dated signals

Offline, synthetic milestone 3 example. The companion Python workflow contains the explicit source preparation, training-only model fits and calls to `research_toolkit`. No private assignment data is used. Prediction loss and portfolio performance measure different things.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src/research_toolkit").exists():
    root = root.parent
sys.path.insert(0, str(root))
import research_toolkit as rt
from examples.chronological_research import run_example

# Session counts, one-session feature lag, one-session boundary gaps.
outputs = run_example(train_sessions=80, validation_sessions=60, test_sessions=80,
                      lag=1, gap_sessions=1, initial_capital=100_000.,
                      annualization=252, rebalance="on_change")

## Sample boundaries and fitted transforms

Each partition ends at its outcome/publication cutoff. Labels crossing that boundary are purged, and only retained training rows fit the standardizer.

In [ ]:
outputs["split"].metadata["boundaries"]

In [ ]:
outputs["split"].excluded.group_by("partition", "reason").len().sort("partition", "reason")

In [ ]:
outputs["split"].transforms

## Validation selection and reserved final test

The validation table selects a predictor; only that predictor is evaluated on test. The source code fixes the allocation rule and costs before test reporting. Preserve the audit when resuming research; further test-based choices require exploratory status.

In [ ]:
outputs["selection"].scores

In [ ]:
outputs["evaluation"].scores

In [ ]:
outputs["evaluation"].audit

## Execution and accounting

Positive forecasts activate equal capital slots, and inactive slots stay in cash. `on_change` sends a new target only when the instruction changes. Next-session-close fills do not earn the move into their own execution close. Ordinary commissions, spread and impact are modeled explicitly.

In [ ]:
outputs["result"].require_complete()
outputs["result"].signal_audit.head(10)

In [ ]:
outputs["result"].trades.head(10)

In [ ]:
outputs["report"].summary

In [ ]:
fig, ax = rt.plots.equity(outputs["report"])

In [ ]:
fig, ax = rt.plots.allocation(outputs["report"])

In [ ]:
fig, ax = rt.plots.drawdown(outputs["report"])

## Interpretation

In the inspected default synthetic run, the lagged linear predictor had lower validation MSE than the training-mean baseline. The portfolio finished with positive net P&L and no risky exposure at the final close. These observations describe this deterministic example; they do not establish out-of-sample market profitability. Changing a model or allocation rule after inspecting these test outputs makes subsequent use exploratory. Write conclusions for any changed configuration only after inspecting its own tables, audit and plots.